In [ ]:
# Example 1: Partition keys and container definitions
from azure.cosmos import PartitionKey
from azure.cosmos.documents import DataType, IndexingMode, IndexKind

single = PartitionKey(path="/category")
assert dict(single)["paths"] == ["/category"] and dict(single)["kind"] == "Hash", dict(single)

# A hierarchical partition key spreads one logical partition over several paths
hierarchical = PartitionKey(path=["/tenantId", "/userId"], kind="MultiHash")
assert dict(hierarchical)["paths"] == ["/tenantId", "/userId"]
assert dict(hierarchical)["kind"] == "MultiHash"

container_definition = {
    "id": "products",
    "partitionKey": dict(single),
    "indexingPolicy": {
        "indexingMode": IndexingMode.Consistent,
        "automatic": True,
        "includedPaths": [{"path": "/*"}],
        "excludedPaths": [{"path": "/description/*"}],
    },
}
# The SDK's enum members are the literal strings the service expects in a policy document
assert container_definition["indexingPolicy"]["indexingMode"] == "consistent"
assert (IndexingMode.Lazy, IndexingMode.NoIndex) == ("lazy", "none")
assert (IndexKind.Hash, IndexKind.Range) == ("Hash", "Range")
assert (DataType.String, DataType.Number, DataType.Point) == ("String", "Number", "Point")
print("partition key:", dict(single))
print("hierarchical key:", dict(hierarchical))

In [ ]:
# Example 2: Throughput settings, status codes and error types
from azure.core.exceptions import HttpResponseError, ResourceExistsError, ResourceNotFoundError
from azure.cosmos import ThroughputProperties, exceptions, http_constants

manual = ThroughputProperties(offer_throughput=400)
autoscale = ThroughputProperties(auto_scale_max_throughput=4000, auto_scale_increment_percent=0)
assert manual.offer_throughput == 400 and manual.auto_scale_max_throughput is None
assert autoscale.auto_scale_max_throughput == 4000 and autoscale.offer_throughput is None

codes = http_constants.StatusCodes
assert (codes.NOT_FOUND, codes.CONFLICT, codes.TOO_MANY_REQUESTS) == (404, 409, 429)

# Cosmos errors are azure-core HTTP errors, so generic Azure error handlers catch them too
assert issubclass(exceptions.CosmosResourceNotFoundError, ResourceNotFoundError)
assert issubclass(exceptions.CosmosResourceExistsError, ResourceExistsError)
assert issubclass(exceptions.CosmosHttpResponseError, HttpResponseError)
try:
    raise exceptions.CosmosResourceExistsError(status_code=codes.CONFLICT, message="Entity with the specified id already exists.")
except HttpResponseError as err:
    assert err.status_code == 409, err.status_code
    print("caught:", type(err).__name__, err.status_code)

In [ ]:
# Example 3: ConnectionPolicy, consistency levels and partition kinds
from azure.cosmos.documents import ConnectionMode, ConnectionPolicy, ConsistencyLevel, PartitionKind

policy = ConnectionPolicy()
# The Python SDK connects in gateway mode: HTTPS requests through the account endpoint
assert policy.ConnectionMode == ConnectionMode.Gateway
assert policy.EnableEndpointDiscovery is True
assert isinstance(policy.RequestTimeout, (int, float)) and policy.RequestTimeout > 0
policy.RequestTimeout = 10
policy.PreferredLocations = ["West US 2", "East US"]
assert (policy.RequestTimeout, policy.PreferredLocations) == (10, ["West US 2", "East US"])

levels = {name for name in vars(ConsistencyLevel) if not name.startswith("_")}
assert levels == {"Strong", "BoundedStaleness", "Session", "ConsistentPrefix", "Eventual"}, levels
assert ConsistencyLevel.Session == "Session"
assert (PartitionKind.Hash, PartitionKind.MultiHash) == ("Hash", "MultiHash")
print("connection mode:", policy.ConnectionMode, "| request timeout:", policy.RequestTimeout, "s")
print("consistency levels:", sorted(levels))

In [ ]:
# Example 4: Read, upsert and query items through a stub transport
# CosmosClient runs its normal request pipeline; CannedAdapter answers each HTTP request from the
# in-memory account below instead of the network. The key is a made-up base64 string.
import base64
import http
import io
import json

import requests
from azure.core.pipeline.transport import RequestsTransport
from azure.cosmos import CosmosClient


class CannedAdapter(requests.adapters.BaseAdapter):
    def __init__(self, handler):
        super().__init__()
        self.handler = handler  # request -> (status code, JSON payload, extra headers)
        self.requests = []

    def send(self, request, **kwargs):
        self.requests.append(request)
        status_code, payload, headers = self.handler(request)
        body = json.dumps(payload).encode()
        response = requests.Response()
        response.status_code = status_code
        response.reason = http.HTTPStatus(status_code).phrase
        response.headers = requests.structures.CaseInsensitiveDict(
            {"Content-Type": "application/json", "Content-Length": str(len(body)), **headers})
        response.raw = io.BytesIO(body)
        response.url = request.url
        response.request = request
        return response

    def close(self):
        pass


ENDPOINT = "https://example-account.documents.azure.com:443/"
LOCATION = {"name": "West US 2", "databaseAccountEndpoint": ENDPOINT}
ACCOUNT = {"id": "example-account", "_rid": "example-account.documents.azure.com", "_dbs": "//dbs/",
           "writableLocations": [LOCATION], "readableLocations": [LOCATION],
           "enableMultipleWriteLocations": False,
           "userConsistencyPolicy": {"defaultConsistencyLevel": "Session"},
           "userReplicationPolicy": {"minReplicaSetSize": 3, "maxReplicasetSize": 4},
           "systemReplicationPolicy": {"minReplicaSetSize": 3, "maxReplicasetSize": 4},
           "readPolicy": {"primaryReadCoefficient": 1, "secondaryReadCoefficient": 1},
           "queryEngineConfiguration": "{}"}
CONTAINER = {"id": "products", "_rid": "cHJvZA==", "_self": "dbs/c2hvcA==/colls/cHJvZA==/",
             "partitionKey": {"paths": ["/category"], "kind": "Hash", "version": 2}}
items = {"item-1": {"id": "item-1", "category": "electronics", "name": "laptop", "price": 999.0}}


def cosmos_service(request):
    path = request.path_url.split("?")[0].strip("/")
    if path == "":
        return 200, ACCOUNT, {}
    if path == "dbs/shop/colls/products":
        return 200, CONTAINER, {}
    if path == "dbs/shop/colls/products/docs" and request.headers.get("x-ms-documentdb-isquery"):
        # The stub answers the one query this example sends: items in a partition above a price
        query = json.loads(request.body)
        (partition,) = json.loads(request.headers["x-ms-documentdb-partitionkey"])
        floor = {p["name"]: p["value"] for p in query["parameters"]}["@min_price"]
        docs = [d for d in items.values() if d["category"] == partition and d["price"] > floor]
        return 200, {"_rid": CONTAINER["_rid"], "Documents": docs, "_count": len(docs)}, {}
    if path == "dbs/shop/colls/products/docs" and request.method == "POST":
        doc = json.loads(request.body)
        items[doc["id"]] = doc
        return 201, doc, {"x-ms-request-charge": "6.2"}
    item_id = path.rsplit("/", 1)[-1]
    if path.startswith("dbs/shop/colls/products/docs/") and item_id in items:
        return 200, items[item_id], {"x-ms-request-charge": "1"}
    return 404, {"code": "NotFound", "message": f"Resource {path} not found"}, {}


adapter = CannedAdapter(cosmos_service)
session = requests.Session()
session.mount("https://", adapter)
client = CosmosClient(ENDPOINT, credential=base64.b64encode(b"example-key").decode(),
                      transport=RequestsTransport(session=session))
container = client.get_database_client("shop").get_container_client("products")

laptop = container.read_item("item-1", partition_key="electronics")
assert laptop["name"] == "laptop" and laptop["price"] == 999.0

container.upsert_item({"id": "item-2", "category": "electronics", "name": "monitor", "price": 199.0})
container.upsert_item({"id": "item-3", "category": "electronics", "name": "cable", "price": 9.0})
assert sorted(items) == ["item-1", "item-2", "item-3"]

results = list(container.query_items(
    "SELECT * FROM c WHERE c.price > @min_price",
    parameters=[{"name": "@min_price", "value": 100}],
    partition_key="electronics",
))
assert sorted(item["name"] for item in results) == ["laptop", "monitor"], results
print("query returned:", sorted(item["name"] for item in results))

In [ ]:
# Example 5: A missing item raises CosmosResourceNotFoundError, and requests carry the key and partition
# Uses `container` and `adapter` from Example 4.
import json
from urllib.parse import unquote

from azure.cosmos import exceptions

try:
    container.read_item("no-such-item", partition_key="electronics")
except exceptions.CosmosResourceNotFoundError as err:
    assert err.status_code == 404, err.status_code
    print("missing item ->", type(err).__name__, err.status_code)
else:
    raise AssertionError("read_item() for a missing id did not raise CosmosResourceNotFoundError")

item_reads = [r for r in adapter.requests if r.method == "GET" and "/docs/" in r.path_url]
assert item_reads, "no item reads were sent"
for request in item_reads:
    # Master-key requests carry a signed authorization header and the partition key as a JSON array
    assert unquote(request.headers["authorization"]).startswith("type=master&"), request.headers["authorization"][:20]
    assert json.loads(request.headers["x-ms-documentdb-partitionkey"]) == ["electronics"]
print(len(adapter.requests), "requests answered by the stub")